# 🏦 PRCP-1000: Portuguese Bank Marketing — Term Deposit Prediction
**Internship Project | Data Science & Machine Learning**

---
| Task | Description |
|------|-------------|
| Task 1 | Complete Exploratory Data Analysis (EDA) Report |
| Task 2 | Build Classification Model to predict product purchase |
| Task 3 | Business Suggestions to improve customer conversion |
| Extra  | Model Comparison Report + Challenges Faced |

**Dataset:** Bank Additional Full Dataset — 21 features, ~41,188 records  
**Domain:** Finance / Banking  
**Target Variable:** `y` — Has the client subscribed a term deposit? (`yes` / `no`)  
**Problem Type:** Binary Classification with Class Imbalance

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, ConfusionMatrixDisplay
)
from sklearn.utils import resample

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('All libraries imported successfully!')

## Step 2: Load Dataset

In [ ]:
import zipfile, os, io

def load_bank_data():
    # Option 1: Direct CSV paths (original dataset)
    for path in [
        'bank-additional/bank-additional-full.csv',
        'bank-additional-full.csv',
        'bank_full.csv'
    ]:
        if os.path.exists(path):
            sep = ';' if path.endswith('bank-additional-full.csv') else ','
            return pd.read_csv(path, sep=sep)

    # Option 2: Extract from zip
    for fname in os.listdir('.'):
        if fname.endswith('.zip'):
            with zipfile.ZipFile(fname, 'r') as z:
                z.extractall('bank_data')
            for root, dirs, files in os.walk('bank_data'):
                for f in files:
                    if 'full' in f and f.endswith('.csv'):
                        return pd.read_csv(os.path.join(root, f), sep=';')

    # Option 3: Colab upload
    try:
        from google.colab import files
        print('Upload your zip or CSV file:')
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        if fname.endswith('.zip'):
            with zipfile.ZipFile(fname, 'r') as z:
                z.extractall('bank_data')
            for root, dirs, flist in os.walk('bank_data'):
                for f in flist:
                    if 'full' in f and f.endswith('.csv'):
                        return pd.read_csv(os.path.join(root, f), sep=';')
        else:
            sep = ';' if 'additional' in fname else ','
            return pd.read_csv(io.BytesIO(uploaded[fname]), sep=sep)
    except:
        pass
    return None

df = load_bank_data()

if df is None:
    print('Using built-in fallback dataset...')
    df = pd.read_csv('bank_full.csv')

# Standardize column names
df.columns = df.columns.str.strip()
print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Target distribution:')
print(df['y'].value_counts())
print(f"\nClass imbalance ratio: {(df['y']=='no').sum() / (df['y']=='yes').sum():.1f}:1")
df.head()

---
# TASK 1: Data Analysis Report

### 1.1 Dataset Overview & Data Types

In [ ]:
print('SHAPE:', df.shape)
print('\nDATA TYPES:')
print(df.dtypes)
print(f'\nDUPLICATES: {df.duplicated().sum()}')
print('\nNUMERICAL:', df.select_dtypes(include=np.number).columns.tolist())
print('\nCATEGORICAL:', df.select_dtypes(include='object').columns.tolist())

### 1.2 Statistical Summary

In [ ]:
print('=== NUMERICAL FEATURES SUMMARY ===')
display(df.describe())
print('\n=== CATEGORICAL FEATURES SUMMARY ===')
display(df.describe(include='object'))

### 1.3 Missing Value Analysis

In [ ]:
# Check actual NaN + check 'unknown' values
missing = df.isnull().sum()
unknown_counts = {col: (df[col] == 'unknown').sum()
                  for col in df.select_dtypes(include='object').columns}

print('ACTUAL NULL VALUES:')
print(missing[missing > 0] if missing.sum() > 0 else '  None found!')

print('\nUNKNOWN VALUES (categorical):')
uk_df = pd.DataFrame({'Unknown Count': unknown_counts,
                       'Unknown %': {k: round(v/len(df)*100,2) for k,v in unknown_counts.items()}})
uk_df = uk_df[uk_df['Unknown Count'] > 0].sort_values('Unknown Count', ascending=False)
print(uk_df)

if len(uk_df) > 0:
    plt.figure(figsize=(10, 4))
    sns.barplot(x=uk_df.index, y=uk_df['Unknown %'], palette='Oranges_r')
    plt.title('Unknown Values (%) by Column', fontsize=13, fontweight='bold')
    plt.xticks(rotation=30)
    plt.ylabel('Unknown %')
    plt.tight_layout()
    plt.show()

### 1.4 Target Variable — Class Imbalance Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['y'].value_counts()
colors = ['#e74c3c', '#2ecc71']

axes[0].pie(counts.values, labels=counts.index,
            autopct='%1.1f%%', colors=colors,
            startangle=90, explode=[0.05, 0], shadow=True)
axes[0].set_title('Target Class Distribution', fontweight='bold', fontsize=13)

bars = axes[1].bar(counts.index, counts.values, color=colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('Count: Subscribed vs Not Subscribed', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Count')

plt.suptitle('Term Deposit Subscription — Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

yes_pct = (df['y']=='yes').mean() * 100
print(f'Subscribed (yes) : {counts["yes"]:,}  ({yes_pct:.1f}%)')
print(f'Not Subscribed   : {counts["no"]:,}  ({100-yes_pct:.1f}%)')
print(f'Imbalance Ratio  : {counts["no"]/counts["yes"]:.1f} : 1  — HIGHLY IMBALANCED!')

### 1.5 Job & Education Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

job_sub = df.groupby('job')['y'].apply(lambda x: (x=='yes').mean()*100).sort_values(ascending=False)
sns.barplot(x=job_sub.values, y=job_sub.index, palette='viridis', ax=axes[0])
axes[0].set_title('Subscription Rate (%) by Job Type', fontweight='bold')
axes[0].set_xlabel('Subscription Rate (%)')
for i, v in enumerate(job_sub.values):
    axes[0].text(v+0.2, i, f'{v:.1f}%', va='center', fontsize=9)

edu_sub = df.groupby('education')['y'].apply(lambda x: (x=='yes').mean()*100).sort_values(ascending=False)
sns.barplot(x=edu_sub.values, y=edu_sub.index, palette='coolwarm_r', ax=axes[1])
axes[1].set_title('Subscription Rate (%) by Education Level', fontweight='bold')
axes[1].set_xlabel('Subscription Rate (%)')

plt.suptitle('Subscription Rate by Job & Education', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.6 Age Distribution & Subscription

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for outcome, color, label in [('yes','#2ecc71','Subscribed'), ('no','#e74c3c','Not Subscribed')]:
    sns.kdeplot(df[df['y']==outcome]['age'], ax=axes[0],
                color=color, label=label, fill=True, alpha=0.4)
axes[0].set_title('Age Distribution by Subscription', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].legend()

df['Age_Group'] = pd.cut(df['age'], bins=[17,25,35,45,55,65,100],
                          labels=['18-25','26-35','36-45','46-55','56-65','65+'])
age_sub = df.groupby('Age_Group', observed=True)['y'].apply(
    lambda x: (x=='yes').mean()*100).reset_index()
sns.barplot(data=age_sub, x='Age_Group', y='y', palette='YlOrRd_r', ax=axes[1])
axes[1].set_title('Subscription Rate (%) by Age Group', fontweight='bold')
axes[1].set_ylabel('Subscription Rate (%)')
axes[1].set_xlabel('Age Group')

plt.suptitle('Age vs Term Deposit Subscription', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.7 Call Duration, Campaign & Month Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Duration
for outcome, color, label in [('yes','#2ecc71','Subscribed'), ('no','#e74c3c','Not Subscribed')]:
    sns.kdeplot(df[df['y']==outcome]['duration'], ax=axes[0],
                color=color, label=label, fill=True, alpha=0.4)
axes[0].set_title('Call Duration vs Subscription', fontweight='bold')
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_xlim(0, 2000)
axes[0].legend()

# Month
month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
month_order = [m for m in month_order if m in df['month'].unique()]
month_sub = df.groupby('month')['y'].apply(lambda x: (x=='yes').mean()*100)
month_sub = month_sub.reindex(month_order).dropna()
axes[1].bar(month_sub.index, month_sub.values,
            color=['#e74c3c' if v < month_sub.mean() else '#2ecc71' for v in month_sub.values],
            edgecolor='white')
axes[1].axhline(month_sub.mean(), color='navy', linestyle='--', lw=1.5, label='Avg')
axes[1].set_title('Subscription Rate (%) by Month', fontweight='bold')
axes[1].set_ylabel('Subscription Rate (%)')
axes[1].legend()

# Campaign contacts
camp_data = df[df['campaign'] <= 10].copy()
camp_sub = camp_data.groupby('campaign')['y'].apply(lambda x: (x=='yes').mean()*100)
axes[2].plot(camp_sub.index, camp_sub.values, marker='o', color='steelblue', lw=2)
axes[2].fill_between(camp_sub.index, camp_sub.values, alpha=0.2, color='steelblue')
axes[2].set_title('Subscription Rate vs Campaign Contacts', fontweight='bold')
axes[2].set_xlabel('Number of Contacts')
axes[2].set_ylabel('Subscription Rate (%)')

plt.suptitle('Call Duration, Month & Campaign Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.8 Previous Campaign Outcome & Contact Type

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# poutcome
pout_sub = df.groupby('poutcome')['y'].apply(lambda x: (x=='yes').mean()*100).sort_values(ascending=False)
sns.barplot(x=pout_sub.index, y=pout_sub.values,
            palette=['#2ecc71','#e74c3c','#3498db'], ax=axes[0])
axes[0].set_title('Subscription Rate by\nPrevious Campaign Outcome', fontweight='bold')
axes[0].set_ylabel('Subscription Rate (%)')
for i, v in enumerate(pout_sub.values):
    axes[0].text(i, v+0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# contact type
contact_sub = df.groupby('contact')['y'].apply(lambda x: (x=='yes').mean()*100)
sns.barplot(x=contact_sub.index, y=contact_sub.values,
            palette=['#9b59b6','#1abc9c'], ax=axes[1])
axes[1].set_title('Subscription Rate by Contact Type', fontweight='bold')
axes[1].set_ylabel('Subscription Rate (%)')
for i, v in enumerate(contact_sub.values):
    axes[1].text(i, v+0.2, f'{v:.1f}%', ha='center', fontweight='bold')

# marital + housing
marital_sub = df.groupby('marital')['y'].apply(lambda x: (x=='yes').mean()*100).sort_values(ascending=False)
sns.barplot(x=marital_sub.index, y=marital_sub.values, palette='Set2', ax=axes[2])
axes[2].set_title('Subscription Rate by Marital Status', fontweight='bold')
axes[2].set_ylabel('Subscription Rate (%)')

plt.suptitle('Previous Outcome, Contact Type & Marital Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.9 Economic Indicators Analysis

In [ ]:
eco_cols = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
eco_labels = ['Emp Variation Rate', 'Consumer Price Idx', 'Consumer Conf Idx',
               'Euribor 3M Rate', 'Nr Employed']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, (col, label) in enumerate(zip(eco_cols, eco_labels)):
    if col in df.columns:
        sns.boxplot(data=df, x='y', y=col, palette=['#e74c3c','#2ecc71'], ax=axes[i])
        axes[i].set_title(label, fontweight='bold', fontsize=9)
        axes[i].set_xlabel('Subscribed?')

plt.suptitle('Economic Indicators vs Subscription Decision', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Mean economic indicators by subscription:')
print(df.groupby('y')[eco_cols].mean().round(3))

### 1.10 Correlation Heatmap (Numerical Features)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
df_corr = df[num_cols].copy()
df_corr['y_enc'] = (df['y'] == 'yes').astype(int)

corr = df_corr.corr()
plt.figure(figsize=(13, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, annot_kws={'size': 8}, vmin=-1, vmax=1)
plt.title('Correlation Matrix — Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlation with Target (y):')
target_corr = corr['y_enc'].drop('y_enc').sort_values(ascending=False)
for feat, val in target_corr.items():
    bar = '█' * int(abs(val) * 30)
    print(f'  {feat:<20} {val:+.4f}  {bar}')

---
# TASK 2: Data Preprocessing & Model Building

### 2.1 Preprocessing — Encode Features

In [ ]:
df_model = df.copy()

# Drop helper columns from EDA
for col in ['Age_Group']:
    if col in df_model.columns:
        df_model.drop(col, axis=1, inplace=True)

# NOTE: Dropping 'duration' per project spec - it's a post-call variable
# 'duration' is not known before the call, including it would be data leakage
if 'duration' in df_model.columns:
    df_model.drop('duration', axis=1, inplace=True)
    print('Dropped duration (post-call variable — data leakage risk)')

# Encode target
df_model['y'] = (df_model['y'] == 'yes').astype(int)

# Label encode all remaining categorical columns
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print(f'Encoding {len(cat_cols)} categorical columns: {cat_cols}')

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    le_dict[col] = le

print(f'\nFinal shape: {df_model.shape}')
print('Encoding complete!')
df_model.head(3)

### 2.2 Handle Class Imbalance — Random Oversampling

In [ ]:
X = df_model.drop('y', axis=1)
y = df_model['y']

# Train-test split BEFORE balancing (prevent data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Before balancing — Train:')
print(f'  Class 0 (no) : {(y_train==0).sum():,}')
print(f'  Class 1 (yes): {(y_train==1).sum():,}')

# Oversample minority class (yes) in training set only
train_df = pd.concat([X_train, y_train], axis=1)
majority  = train_df[train_df['y'] == 0]
minority  = train_df[train_df['y'] == 1]

minority_upsampled = resample(minority,
                               replace=True,
                               n_samples=len(majority),
                               random_state=42)
train_balanced = pd.concat([majority, minority_upsampled]).sample(frac=1, random_state=42)

X_train_bal = train_balanced.drop('y', axis=1)
y_train_bal = train_balanced['y']

print(f'\nAfter balancing — Train:')
print(f'  Class 0 (no) : {(y_train_bal==0).sum():,}')
print(f'  Class 1 (yes): {(y_train_bal==1).sum():,}')

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bal)
X_test_sc  = scaler.transform(X_test)

print(f'\nTest set (original distribution):')
print(f'  Class 0: {(y_test==0).sum():,}  Class 1: {(y_test==1).sum():,}')

### 2.3 Train All 6 Classification Models

In [ ]:
models = {
    'Logistic Regression'  : (LogisticRegression(max_iter=1000, random_state=42), True),
    'K-Nearest Neighbors'  : (KNeighborsClassifier(n_neighbors=5),                True),
    'Naive Bayes'          : (GaussianNB(),                                        True),
    'Decision Tree'        : (DecisionTreeClassifier(max_depth=8, random_state=42), False),
    'Random Forest'        : (RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), False),
    'Gradient Boosting'    : (GradientBoostingClassifier(n_estimators=100, random_state=42), False),
}

results = []
trained_models = {}

print(f"{'Model':<24} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'ROC-AUC':>9}")
print('-' * 75)

for name, (model, use_scaled) in models.items():
    Xtr = X_train_sc if use_scaled else X_train_bal
    Xte = X_test_sc  if use_scaled else X_test

    model.fit(Xtr, y_train_bal)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:,1] if hasattr(model,'predict_proba') else y_pred

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_prob)

    results.append({'Model': name, 'Accuracy': round(acc,4),
                    'Precision': round(prec,4), 'Recall': round(rec,4),
                    'F1 Score': round(f1,4), 'ROC-AUC': round(auc,4)})
    trained_models[name] = (model, use_scaled)
    print(f"{name:<24} {acc:>9.4f} {prec:>10.4f} {rec:>8.4f} {f1:>8.4f} {auc:>9.4f}")

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print('\nAll 6 models trained!')

---
# MODEL COMPARISON REPORT

In [ ]:
print('=' * 75)
print('          MODEL COMPARISON REPORT — PRCP-1000')
print('=' * 75)
print(results_df.to_string(index=False))
print('=' * 75)
best = results_df.iloc[0]
print(f"\nBEST MODEL FOR PRODUCTION: {best['Model']}")
print(f"  ROC-AUC   : {best['ROC-AUC']}")
print(f"  F1 Score  : {best['F1 Score']}")
print(f"  Recall    : {best['Recall']} (ability to find actual subscribers)")
print(f"  Precision : {best['Precision']}")

In [ ]:
metrics = ['ROC-AUC', 'F1 Score', 'Recall', 'Precision', 'Accuracy']
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
colors = ['#27ae60' if i==0 else '#3498db' for i in range(len(results_df))]

for ax, metric in zip(axes, metrics):
    sorted_df = results_df.sort_values(metric, ascending=True)
    c = ['#27ae60' if m == results_df.iloc[0]['Model'] else '#3498db'
         for m in sorted_df['Model']]
    bars = ax.barh(sorted_df['Model'], sorted_df[metric], color=c)
    ax.set_title(metric, fontweight='bold', fontsize=10)
    ax.set_xlim(0, 1.1)
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(val+0.01, bar.get_y()+bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)

plt.suptitle('Model Comparison — All Metrics  (Green = Best by ROC-AUC)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### ROC Curves — All Models

In [ ]:
plt.figure(figsize=(10, 7))
colors_roc = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']

for (name, (model, use_sc)), color in zip(trained_models.items(), colors_roc):
    Xte = X_test_sc if use_sc else X_test
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(Xte)[:,1]
    else:
        y_prob = model.predict(Xte)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

plt.plot([0,1],[0,1],'k--', lw=1, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All 6 Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Confusion Matrix & Classification Report (Best Model)

In [ ]:
best_name = results_df.iloc[0]['Model']
best_obj, use_sc = trained_models[best_name]
Xte_use = X_test_sc if use_sc else X_test
y_pred_best = best_obj.predict(Xte_use)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(cm, display_labels=['No (0)','Yes (1)'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Confusion Matrix — {best_name}', fontweight='bold')

# Feature importance
if hasattr(best_obj, 'feature_importances_'):
    fi = pd.Series(best_obj.feature_importances_, index=X.columns).sort_values(ascending=True).tail(15)
    fi.plot(kind='barh', color='steelblue', edgecolor='white', ax=axes[1])
    axes[1].set_title(f'Feature Importances — {best_name}', fontweight='bold')
    axes[1].set_xlabel('Importance Score')
elif hasattr(best_obj, 'coef_'):
    coef = pd.Series(np.abs(best_obj.coef_[0]), index=X.columns).sort_values(ascending=True).tail(15)
    coef.plot(kind='barh', color='steelblue', edgecolor='white', ax=axes[1])
    axes[1].set_title(f'Feature Coefficients — {best_name}', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nCLASSIFICATION REPORT — {best_name}:')
print(classification_report(y_test, y_pred_best, target_names=['No','Yes']))

---
# TASK 3: Business Suggestions to Improve Customer Conversion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Best months to call
month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
orig_df = df.copy() if 'y' not in df.columns or df['y'].dtype != int else df

# Use original df for EDA
df_orig = df.copy()
month_sub2 = df_orig.groupby('month')['y'].apply(
    lambda x: (x == 'yes').mean() * 100 if x.dtype == object else x.mean() * 100
).reindex([m for m in month_order if m in df_orig['month'].unique()])

top_months = month_sub2.nlargest(4).index.tolist()
bar_colors = ['#27ae60' if m in top_months else '#bdc3c7' for m in month_sub2.index]
axes[0].bar(month_sub2.index, month_sub2.values, color=bar_colors, edgecolor='white')
axes[0].set_title('Best Months to Call\n(Green = Top 4)', fontweight='bold')
axes[0].set_ylabel('Subscription Rate (%)')
axes[0].tick_params(axis='x', rotation=45)

# Best jobs to target
job_sub2 = df_orig.groupby('job')['y'].apply(
    lambda x: (x=='yes').mean()*100 if x.dtype==object else x.mean()*100
).sort_values(ascending=False)
colors_j = ['#27ae60' if i<3 else '#bdc3c7' for i in range(len(job_sub2))]
sns.barplot(x=job_sub2.values, y=job_sub2.index, palette=colors_j, ax=axes[1])
axes[1].set_title('Job Types — Top Targets\n(Green = Top 3)', fontweight='bold')
axes[1].set_xlabel('Subscription Rate (%)')

# Age groups to prioritize
age_sub2 = df_orig.groupby(pd.cut(df_orig['age'], bins=[17,25,35,45,55,65,100],
    labels=['18-25','26-35','36-45','46-55','56-65','65+']), observed=True)['y'].apply(
    lambda x: (x=='yes').mean()*100 if x.dtype==object else x.mean()*100
).sort_values(ascending=False)
colors_a = ['#27ae60' if i<2 else '#bdc3c7' for i in range(len(age_sub2))]
sns.barplot(x=age_sub2.values, y=age_sub2.index.astype(str),
            palette=colors_a, ax=axes[2])
axes[2].set_title('Age Groups to Prioritize\n(Green = Top 2)', fontweight='bold')
axes[2].set_xlabel('Subscription Rate (%)')

plt.suptitle('Business Targeting Strategy — Data-Driven Insights', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
suggestions = """
====================================================================
     TASK 3: BUSINESS SUGGESTIONS — PORTUGUESE BANK MARKETING
====================================================================

STRATEGY 1: TARGET THE RIGHT CUSTOMER PROFILE
  High-Value Targets:
  - Students & Retired individuals show highest subscription rates
  - Age groups 18-25 and 65+ are most receptive
  - Single customers convert better than married
  - University-educated customers are more likely to subscribe
  Action: Prioritize calling lists with these demographics first.

STRATEGY 2: OPTIMIZE CALLING MONTHS & TIMING
  Best months: March, September, October, December
  Why: These months show 2-3x higher subscription rates than May
  Action: Increase campaign intensity in Q4 and early Q1.
          Reduce calls in May-August (high volume, low conversion).

STRATEGY 3: FOCUS ON CALL QUALITY OVER QUANTITY
  - Longer call duration strongly predicts subscription
  - Customers who speak >3 mins are far more likely to subscribe
  - Calling the same customer >3 times has diminishing returns
  Action: Train agents to have meaningful conversations (not scripts).
          Set a max 3-contact rule per customer per campaign.

STRATEGY 4: LEVERAGE PREVIOUS CAMPAIGN SUCCESS
  - Customers with previous 'success' outcome convert at 64%+ rate
  - 'Failure' from previous campaign still converts at ~12%
  Action: Always re-contact previous 'success' customers first.
          Build a loyalty/retention pool from repeat subscribers.

STRATEGY 5: PREFER CELLULAR OVER TELEPHONE
  - Cellular contact has significantly higher conversion rate
  - Telephone landline calls have lower engagement
  Action: Prioritize mobile numbers in contact lists.
          Invest in mobile contact data enrichment.

STRATEGY 6: USE THE ML MODEL FOR LEAD SCORING
  - The Gradient Boosting / Random Forest model predicts subscription
    with high ROC-AUC (meaningful prediction above random chance)
  - Score each lead BEFORE calling
  Action: Call high-score leads first each day.
          Set a probability threshold (e.g. >0.4) to decide who to call.

STRATEGY 7: MONITOR ECONOMIC INDICATORS
  - Low euribor3m rate & negative emp.var.rate correlate with more subscriptions
  - Customers subscribe more during economic uncertainty
  Action: Run intensive campaigns when euribor rate is below 2%.
          Brief management team monthly on economic context.

STRATEGY 8: REDUCE UNKNOWN DATA QUALITY
  - 'unknown' in job, education, default fields reduces model accuracy
  Action: Improve CRM data completeness.
          Incentivize customers to update profile information.

===================================================================="""
print(suggestions)

---
# CHALLENGES FACED & TECHNIQUES USED

In [ ]:
challenges = """
CHALLENGES FACED AND TECHNIQUES USED — PRCP-1000
====================================================================

CHALLENGE 1: Severe Class Imbalance (~89% No vs ~11% Yes)
  Details  : Only 11% of customers subscribed. Training a model on
             imbalanced data causes it to predict 'no' for everything
             and still achieve 89% accuracy — but 0% recall on 'yes'.
  Technique: Random Oversampling of the minority class (yes) in the
             training set only (AFTER train-test split).
  Reason   : Balancing ONLY the training set prevents data leakage.
             Oversampling preserves all real 'yes' patterns.
             SMOTE was also an option but resample is simpler here.

CHALLENGE 2: Data Leakage Risk — 'duration' Feature
  Details  : Duration (call length) strongly predicts subscription
             but is NOT known before the call is made. Including it
             creates an unrealistic 'future-peaking' model.
  Technique: Removed 'duration' from the feature set for modeling.
  Reason   : The project spec explicitly warns about this. A real
             production model must predict BEFORE the call happens.

CHALLENGE 3: 'unknown' Values in Categorical Columns
  Details  : Several columns (job, education, default, housing, loan)
             have 'unknown' as a category — not truly missing but
             representing uncertainty in the data.
  Technique: Kept 'unknown' as its own category and label-encoded it.
  Reason   : Dropping rows with 'unknown' would remove valid records.
             The model learns that 'unknown' itself is a signal.

CHALLENGE 4: Large Dataset (41,188 rows)
  Details  : Training time for complex models like Gradient Boosting
             is significant on 80K+ balanced training rows.
  Technique: n_estimators=100 (not too many), n_jobs=-1 for RF.
             KNN limited to k=5 to reduce search complexity.
  Reason   : Balances accuracy vs training speed for submission.

CHALLENGE 5: Choosing the Right Evaluation Metric
  Details  : Accuracy is misleading for imbalanced data. High accuracy
             does NOT mean the model is finding subscribers correctly.
  Technique: Prioritized ROC-AUC and F1-Score as primary metrics.
             Recall is especially important — missing a subscriber
             (false negative) costs the bank revenue.
  Reason   : ROC-AUC measures overall discrimination ability.
             F1 balances precision and recall.
             For banking marketing: high recall = don't miss leads.

CHALLENGE 6: High Multicollinearity in Economic Indicators
  Details  : emp.var.rate, euribor3m, nr.employed are all highly
             correlated (r > 0.9) — they move together.
  Technique: Kept all features but used tree-based models as primary.
             Ridge/regularized logistic regression handles multicollinearity.
  Reason   : Tree ensembles are inherently robust to correlated features
             by selecting different split points per tree.

CHALLENGE 7: Stratified Split Required
  Details  : Simple random split could give test set with very few
             'yes' samples — making evaluation unreliable.
  Technique: Used stratify=y in train_test_split.
  Reason   : Ensures the same 11% yes ratio in both train and test sets
             for fair and representative model evaluation.

===================================================================="""
print(challenges)

---
# Final Summary & Conclusion

In [ ]:
print('MODEL PERFORMANCE SUMMARY:')
print(results_df.to_string(index=False))

conclusion = f"""
====================================================================
             SUMMARY & CONCLUSION — PRCP-1000
====================================================================

DATASET
  41,188 bank customer records from May 2008 to November 2010
  21 features: demographics, campaign, economic indicators
  Target: y — Term Deposit Subscription (11% yes, 89% no)
  HIGHLY IMBALANCED classification problem

KEY EDA INSIGHTS
  - Only ~11% of customers subscribed to term deposits
  - Previous campaign success predicts current success strongly
  - Cellular contact outperforms telephone contact
  - March, September, October, December are best months
  - Students and retired customers have highest conversion rates
  - Low euribor rate environment boosts subscriptions
  - More than 3 calls per customer reduces conversion probability

PREPROCESSING STEPS
  - Removed 'duration' (data leakage — post-call variable)
  - Label encoded all categorical features
  - Stratified train-test split (80/20)
  - Random oversampling of minority class in training set only

RECOMMENDED MODEL FOR PRODUCTION: {results_df.iloc[0]['Model']}
  ROC-AUC   : {results_df.iloc[0]['ROC-AUC']}
  F1 Score  : {results_df.iloc[0]['F1 Score']}
  Recall    : {results_df.iloc[0]['Recall']}
  Precision : {results_df.iloc[0]['Precision']}

  Why: Highest ROC-AUC means best overall discrimination between
  subscribers and non-subscribers. High recall ensures the marketing
  team doesn't miss potential customers.

TOP 3 BUSINESS ACTIONS
  1. Score leads with ML model — call high-probability leads first
  2. Focus campaigns on March, Sept, Oct, Dec for best ROI
  3. Cap at 3 calls per customer — more calls hurt conversion

====================================================================
"""
print(conclusion)